# HW3 — Two retrievers and a model: what RAG actually sends

**Serbina Sofiya**
**S23070210**
**CSS4007-ENG-10**
**https://github.com/vesta-12/hw3-vesta-12**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

### A1 — BM25

#### Tokenisation

Following the required rules, I lowercase the text, split on spaces and hyphens,
remove punctuation, and keep stop words.

D1:
large, language, models, can, generate, text, and, answer, questions

|D1| = 9

D2:
retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer

|D2| = 10

D3:
embeddings, represent, text, as, numerical, vectors

|D3| = 6

Query:
how, does, rag, retrieve, information

#### Document lengths

N = 3

|D1| = 9
|D2| = 10
|D3| = 6

avgdl = (9 + 10 + 6) / 3
      = 25 / 3
      = 8.3333

### A1(a) — BM25 without stemming

#### Term frequency

The query terms are:

how, does, rag, retrieve, information

I count only exact token matches.

| Query term | D1 | D2 | D3 |
|---|---:|---:|---:|
| how | 0 | 0 | 0 |
| does | 0 | 0 | 0 |
| rag | 0 | 0 | 0 |
| retrieve | 0 | 0 | 0 |
| information | 0 | 0 | 0 |

D2 contains the words `retrieval` and `retrieves`, but neither is exactly equal to the query token `retrieve`.

Therefore, every query term has frequency 0 in every document.

#### Document frequency and IDF

Document frequency `n(t)` is the number of documents that contain the query term.

Since none of the query terms appears exactly in D1, D2, or D3:

| Query term | n(t) |
|---|---:|
| how | 0 |
| does | 0 |
| rag | 0 |
| retrieve | 0 |
| information | 0 |

The IDF formula is:

\[
idf(t) = \ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
\]

Here, \(N=3\).

For every query term, \(n(t)=0\), so:

\[
idf(t)=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
\]

\[
=\ln\left(\frac{3.5}{0.5}\right)
\]

\[
=\ln(7)
\]

\[
\approx 1.9459
\]

Therefore:

| Query term | n(t) | IDF |
|---|---:|---:|
| how | 0 | 1.9459 |
| does | 0 | 1.9459 |
| rag | 0 | 1.9459 |
| retrieve | 0 | 1.9459 |
| information | 0 | 1.9459 |

#### BM25 scores

The BM25 formula is:

\[
score(q,d)=\sum_{t \in q}
idf(t)\cdot
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}
\]

where:

\[
k_1=1.5,\qquad b=0.75,\qquad avgdl=8.3333
\]

For all query terms in all three documents:

\[
f(t,d)=0
\]

Therefore, the numerator of every term contribution is:

\[
f(t,d)(k_1+1)=0(1.5+1)=0
\]

So every query term contributes 0 to the BM25 score.

For D1:

\[
score(q,D1)=0
\]

For D2:

\[
score(q,D2)=0
\]

For D3:

\[
score(q,D3)=0
\]

| Document | Length | BM25 score |
|---|---:|---:|
| D1 | 9 | 0 |
| D2 | 10 | 0 |
| D3 | 6 | 0 |

#### Ranking

All three documents have the same score:

\[
D1 = D2 = D3
\]

There is no unique first-ranked document.

#### Explanation

At first this result looks surprising because D2 is clearly related to the query.

However, BM25 without stemming uses exact token matches.

The query contains:

`retrieve`

but D2 contains:

`retrieval` and `retrieves`

These are different tokens under the required tokenisation rules.

Therefore, none of the query terms occurs exactly in any document, so all three BM25 scores are 0.

This is why BM25 cannot choose a unique first document in part (a).

### A1(b) — BM25 with stemming

The following stems are applied wherever these words appear:

- retrieve, retrieves, retrieval → `retriev`
- generate, generating, generation → `gener`
- information → `inform`
- does → `doe`

All other words stay unchanged.

#### Stemmed query

Original query:

`how, does, rag, retrieve, information`

After stemming:

`how, doe, rag, retriev, inform`

#### Stemmed documents

D1:

`large, language, models, can, gener, text, and, answer, questions`

D2:

`retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer`

D3:

`embeddings, represent, text, as, numerical, vectors`

The document lengths do not change:

\[
|D1|=9,\qquad |D2|=10,\qquad |D3|=6
\]

and:

\[
avgdl=8.3333
\]

#### Term frequency after stemming

| Query term | D1 | D2 | D3 |
|---|---:|---:|---:|
| how | 0 | 0 | 0 |
| doe | 0 | 0 | 0 |
| rag | 0 | 0 | 0 |
| retriev | 0 | 2 | 0 |
| inform | 0 | 0 | 0 |

The stem `retriev` occurs twice in D2 because both `retrieval` and `retrieves` are converted to the same stem.

#### Document frequency and IDF after stemming

The stemmed query terms are:

`how, doe, rag, retriev, inform`

Their document frequencies are:

| Query term | n(t) |
|---|---:|
| how | 0 |
| doe | 0 |
| rag | 0 |
| retriev | 1 |
| inform | 0 |

Only `retriev` appears in a document, and it appears in D2.

The IDF formula is:

\[
idf(t)=\ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
\]

where:

\[
N=3
\]

For terms with \(n(t)=0\):

\[
idf(t)=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
\]

\[
=\ln(7)
\]

\[
\approx 1.9459
\]

For `retriev`, \(n(t)=1\):

\[
idf(retriev)=\ln\left(\frac{3-1+0.5}{1+0.5}\right)
\]

\[
=\ln\left(\frac{2.5}{1.5}\right)
\]

\[
=\ln(1.6667)
\]

\[
\approx 0.5108
\]

Therefore:

| Query term | n(t) | IDF |
|---|---:|---:|
| how | 0 | 1.9459 |
| doe | 0 | 1.9459 |
| rag | 0 | 1.9459 |
| retriev | 1 | 0.5108 |
| inform | 0 | 1.9459 |

#### BM25 scores after stemming

The BM25 formula is:

\[
score(q,d)=\sum_{t \in q}
idf(t)\cdot
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}
\]

where:

\[
k_1=1.5,\qquad b=0.75,\qquad avgdl=8.3333
\]

For D1 and D3, none of the stemmed query terms occurs, so:

\[
score(q,D1)=0
\]

\[
score(q,D3)=0
\]

For D2, only the stem `retriev` contributes.

For `retriev`:

\[
f(retriev,D2)=2
\]

\[
idf(retriev)=0.5108
\]

First calculate the document length normalization:

\[
1-b+b\frac{|D2|}{avgdl}
\]

\[
=1-0.75+0.75\frac{10}{8.3333}
\]

\[
=0.25+0.75(1.2)
\]

\[
=0.25+0.9
\]

\[
=1.15
\]

Then:

\[
k_1\left(1-b+b\frac{|D2|}{avgdl}\right)
\]

\[
=1.5(1.15)
\]

\[
=1.725
\]

The denominator is:

\[
f(t,d)+1.725
\]

\[
=2+1.725
\]

\[
=3.725
\]

The numerator is:

\[
f(t,d)(k_1+1)
\]

\[
=2(1.5+1)
\]

\[
=2(2.5)
\]

\[
=5
\]

Therefore:

\[
score(q,D2)
=
0.5108\cdot\frac{5}{3.725}
\]

\[
\approx 0.6857
\]

So the BM25 scores are:

| Document | BM25 score |
|---|---:|
| D1 | 0 |
| D2 | 0.6857 |
| D3 | 0 |

#### Ranking

\[
D2 > D1 = D3
\]

Therefore, with stemming, BM25 retrieves D2 first.

#### A1 conclusion

Without stemming, all three documents receive a BM25 score of 0 because none of the query terms matches any document token exactly.

In particular, the query contains `retrieve`, while D2 contains `retrieval` and `retrieves`.

After stemming, these three word forms become the same stem, `retriev`.

As a result, D2 receives a positive BM25 score and is ranked first.

So:

- without stemming: D1 = D2 = D3
- with stemming: D2 > D1 = D3

### A2 — Cosine similarity

The vectors are:

\[
Q=[0.8,0.2,0.5]
\]

\[
D1=[0.7,0.1,0.4]
\]

\[
D2=[0.9,0.3,0.6]
\]

\[
D3=[0.1,0.8,0.2]
\]

Cosine similarity is calculated as:

\[
\cos(Q,D)=\frac{Q\cdot D}{||Q||\cdot||D||}
\]

#### Magnitude of the query

\[
||Q||=\sqrt{0.8^2+0.2^2+0.5^2}
\]

\[
=\sqrt{0.64+0.04+0.25}
\]

\[
=\sqrt{0.93}
\]

\[
\approx 0.9644
\]

#### D1

Dot product:

\[
Q\cdot D1=(0.8)(0.7)+(0.2)(0.1)+(0.5)(0.4)
\]

\[
=0.56+0.02+0.20
\]

\[
=0.78
\]

Magnitude of D1:

\[
||D1||=\sqrt{0.7^2+0.1^2+0.4^2}
\]

\[
=\sqrt{0.49+0.01+0.16}
\]

\[
=\sqrt{0.66}
\]

\[
\approx 0.8124
\]

Cosine similarity:

\[
\cos(Q,D1)=\frac{0.78}{(0.9644)(0.8124)}
\]

\[
\approx 0.9956
\]

#### D2

Dot product:

\[
Q\cdot D2=(0.8)(0.9)+(0.2)(0.3)+(0.5)(0.6)
\]

\[
=0.72+0.06+0.30
\]

\[
=1.08
\]

Magnitude of D2:

\[
||D2||=\sqrt{0.9^2+0.3^2+0.6^2}
\]

\[
=\sqrt{0.81+0.09+0.36}
\]

\[
=\sqrt{1.26}
\]

\[
\approx 1.1225
\]

Cosine similarity:

\[
\cos(Q,D2)=\frac{1.08}{(0.9644)(1.1225)}
\]

\[
\approx 0.9977
\]

#### D3

Dot product:

\[
Q\cdot D3=(0.8)(0.1)+(0.2)(0.8)+(0.5)(0.2)
\]

\[
=0.08+0.16+0.10
\]

\[
=0.34
\]

Magnitude of D3:

\[
||D3||=\sqrt{0.1^2+0.8^2+0.2^2}
\]

\[
=\sqrt{0.01+0.64+0.04}
\]

\[
=\sqrt{0.69}
\]

\[
\approx 0.8307
\]

Cosine similarity:

\[
\cos(Q,D3)=\frac{0.34}{(0.9644)(0.8307)}
\]

\[
\approx 0.4244
\]

#### A2 summary

| Document | Dot product | Magnitude of query | Magnitude of document | Cosine similarity |
|---|---:|---:|---:|---:|
| D1 | 0.78 | 0.9644 | 0.8124 | 0.9956 |
| D2 | 1.08 | 0.9644 | 1.1225 | 0.9977 |
| D3 | 0.34 | 0.9644 | 0.8307 | 0.4244 |

#### Ranking

\[
D2 > D1 > D3
\]

Therefore, D2 is the document most similar to the query.

#### Comparison with BM25

The cosine similarity ranking is:

\[
D2 > D1 > D3
\]

The BM25 ranking without stemming was:

\[
D1 = D2 = D3
\]

The BM25 ranking with stemming was:

\[
D2 > D1 = D3
\]

So cosine similarity agrees with stemmed BM25 that D2 should be ranked first, but cosine similarity also distinguishes D1 from D3.

#### Why BM25 and embedding retrieval disagree

BM25 relies on matching query terms with document terms.

In this example, the query contains the word `retrieve`, but D2 contains `retrieval` and `retrieves`. Without stemming, BM25 treats these as different tokens, so it gives D2 no lexical match.

The embedding vectors can still represent the semantic similarity between the query and D2. D2 describes Retrieval-Augmented Generation and retrieving relevant documents, so its vector is very close to the query vector even though the exact word `retrieve` is not present.

This is why D2 is ranked first by cosine similarity, while BM25 without stemming cannot distinguish between the three documents.

In [1]:
import math

# ---------- A1: BM25 check ----------

k1 = 1.5
b = 0.75

doc_lengths = {
    "D1": 9,
    "D2": 10,
    "D3": 6
}

avgdl = sum(doc_lengths.values()) / len(doc_lengths)

print("A1 BM25 CHECK")
print("avgdl =", avgdl)

# A1(a): without stemming
scores_without_stemming = {
    "D1": 0.0,
    "D2": 0.0,
    "D3": 0.0
}

print("\nWithout stemming:")
for doc, score in scores_without_stemming.items():
    print(doc, "=", score)

# A1(b): with stemming
N = 3
n_retriev = 1
f_retriev_D2 = 2

idf_retriev = math.log(
    (N - n_retriev + 0.5) /
    (n_retriev + 0.5)
)

d2_length = doc_lengths["D2"]

d2_score = (
    idf_retriev
    * f_retriev_D2
    * (k1 + 1)
    /
    (
        f_retriev_D2
        + k1 * (
            1 - b
            + b * d2_length / avgdl
        )
    )
)

print("\nWith stemming:")
print("IDF(retriev) =", round(idf_retriev, 4))
print("D1 =", 0.0)
print("D2 =", round(d2_score, 4))
print("D3 =", 0.0)


# ---------- A2: cosine similarity check ----------

query = [0.8, 0.2, 0.5]

documents = {
    "D1": [0.7, 0.1, 0.4],
    "D2": [0.9, 0.3, 0.6],
    "D3": [0.1, 0.8, 0.2]
}

def magnitude(vector):
    return math.sqrt(sum(x ** 2 for x in vector))

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

query_magnitude = magnitude(query)

print("\nA2 COSINE SIMILARITY CHECK")
print("Query magnitude =", round(query_magnitude, 4))

results = []

for name, vector in documents.items():
    dot = dot_product(query, vector)
    doc_magnitude = magnitude(vector)
    cosine = dot / (query_magnitude * doc_magnitude)

    results.append((name, cosine))

    print(f"\n{name}")
    print("dot product =", round(dot, 4))
    print("document magnitude =", round(doc_magnitude, 4))
    print("cosine similarity =", round(cosine, 4))

print("\nRanking:")
for rank, (name, cosine) in enumerate(
    sorted(results, key=lambda x: x[1], reverse=True),
    start=1
):
    print(rank, name, round(cosine, 4))

A1 BM25 CHECK
avgdl = 8.333333333333334

Without stemming:
D1 = 0.0
D2 = 0.0
D3 = 0.0

With stemming:
IDF(retriev) = 0.5108
D1 = 0.0
D2 = 0.6857
D3 = 0.0

A2 COSINE SIMILARITY CHECK
Query magnitude = 0.9644

D1
dot product = 0.78
document magnitude = 0.8124
cosine similarity = 0.9956

D2
dot product = 1.08
document magnitude = 1.1225
cosine similarity = 0.9977

D3
dot product = 0.34
document magnitude = 0.8307
cosine similarity = 0.4244

Ranking:
1 D2 0.9977
2 D1 0.9956
3 D3 0.4244


## Part B — Library retrievers

### B1 — BM25 with rank_bm25

For BM25 tokenisation I use the following rules:

- lowercase all text;
- extract words with a regular expression;
- keep hyphenated tokens such as `GR-305` together;
- keep stop words;
- do not apply stemming.

I keep hyphenated form codes together because codes such as `GR-305`, `GR-104`,
and `GR-210` can be important exact lexical matches in this corpus.

In [2]:
import json
import re

from rank_bm25 import BM25Okapi


# Load corpus
corpus = []

with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))


# Load fixed queries
with open("data/queries.json", "r", encoding="utf-8") as f:
    query_data = json.load(f)

fixed_queries = query_data["queries"]


print("Number of chunks:", len(corpus))
print("Number of fixed queries:", len(fixed_queries))

print("\nFirst chunk:")
print("ID:", corpus[0]["id"])
print("Title:", corpus[0]["title"])
print("Language:", corpus[0]["lang"])
print("Text:", corpus[0]["text"][:200] + "...")

Number of chunks: 74
Number of fixed queries: 8

First chunk:
ID: KZ-01
Title: Kazakhstan (part 1)
Language: en
Text: Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the n...


In [3]:
def tokenize(text):
    return re.findall(r"\b[\w]+(?:-[\w]+)*\b", text.lower())


# Small check of the tokenizer
examples = [
    "Form GR-305 is used for appeals.",
    "Retrieval-Augmented Generation",
    "Өтінішті қашанға дейін тапсыру керек?"
]

for example in examples:
    print(example)
    print(tokenize(example))
    print()

Form GR-305 is used for appeals.
['form', 'gr-305', 'is', 'used', 'for', 'appeals']

Retrieval-Augmented Generation
['retrieval-augmented', 'generation']

Өтінішті қашанға дейін тапсыру керек?
['өтінішті', 'қашанға', 'дейін', 'тапсыру', 'керек']



In [4]:
tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in corpus
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

print("BM25 index built successfully.")
print("Indexed chunks:", len(tokenized_corpus))

BM25 index built successfully.
Indexed chunks: 74


In [5]:
def first_line(text):
    for line in text.splitlines():
        line = line.strip()
        if line:
            return line
    return ""


def bm25_top_k(query, k=3):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:k]

    results = []

    for rank, i in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "id": corpus[i]["id"],
            "title": corpus[i]["title"],
            "first_line": first_line(corpus[i]["text"]),
            "score": float(scores[i])
        })

    return results

In [6]:
test_query = fixed_queries[0]

print("Query:", test_query["id"])
print(test_query["text"])
print("Relevant:", test_query["relevant"])

results = bm25_top_k(test_query["text"], k=3)

for result in results:
    print(
        f'{result["rank"]}. '
        f'{result["id"]} | '
        f'{result["first_line"]} | '
        f'score = {result["score"]:.4f}'
    )

Query: Q-01
What is form GR-305 used for?
Relevant: ['GO-08', 'GO-12', 'GO-23']
1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | score = 13.3065
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | score = 6.4533
3. GO-06 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email

In [7]:
bm25_results = {}

for query in fixed_queries:
    query_id = query["id"]
    query_text = query["text"]
    relevant = query["relevant"]

    results = bm25_top_k(query_text, k=3)
    bm25_results[query_id] = results

    retrieved_ids = [result["id"] for result in results]
    hit_at_3 = any(chunk_id in relevant for chunk_id in retrieved_ids)

    print("=" * 100)
    print(f'{query_id}: {query_text}')
    print(f'Relevant chunks: {relevant}')
    print()

    for result in results:
        print(
            f'{result["rank"]}. '
            f'{result["id"]} | '
            f'{result["first_line"]} | '
            f'score = {result["score"]:.4f}'
        )

    print()
    print("hit@3:", hit_at_3)
    print()

Q-01: What is form GR-305 used for?
Relevant chunks: ['GO-08', 'GO-12', 'GO-23']

1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | score = 13.3065
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | score = 6.4533
3. GO-06 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an ema

In [8]:
bm25_hits = {}

for query in fixed_queries:
    query_id = query["id"]
    relevant = query["relevant"]

    retrieved_ids = [
        result["id"]
        for result in bm25_results[query_id]
    ]

    bm25_hits[query_id] = any(
        chunk_id in relevant
        for chunk_id in retrieved_ids
    )


print("BM25 hit@3 summary")
print("-" * 30)

for query_id, hit in bm25_hits.items():
    print(f"{query_id}: {hit}")

total_hits = sum(bm25_hits.values())

print()
print(f"Total: {total_hits} / {len(fixed_queries)}")

BM25 hit@3 summary
------------------------------
Q-01: True
Q-02: False
Q-03: True
Q-04: False
Q-05: True
Q-06: True
Q-07: False
Q-08: True

Total: 5 / 8


### B2 — EmbeddingGemma

For semantic retrieval I use `google/embeddinggemma-300m`
with SentenceTransformers.

The corpus chunks are encoded once with `encode_document()`,
while queries are encoded with `encode_query()`.

This distinction is important because EmbeddingGemma uses different
instruction prefixes for documents and queries.

Cosine similarity is then used to rank all corpus chunks for each query.

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | | | |
| Q-10 | | | |

## Results for all ten or more queries: top 3 from each retriever

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | | | | |
| Q-02 | | | | |
| Q-03 | | | | |
| Q-04 | | | | |
| Q-05 | | | | |
| Q-06 | | | | |
| Q-07 | | | | |
| Q-08 | | | | |
| Q-09 | | | | |
| Q-10 | | | | |
| **hit@3 total** | | /10 | | /10 |

**Both retrievers agree:** which query, and why was it easy?

>

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>

**Embeddings did better:** which query, and what could BM25 not match?

>

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

## C1 — the complete prompt, printed before it is sent

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | |
| Question | |
| Retrieved context | |
| **Total input** | |
| **`usage.prompt_tokens` from the response** | |

- Number of retrieved chunks:
- Tokens per chunk:
- Total context size (tokens / characters):

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>

**What exactly does retrieval add to the model's input?**

>

## C2 — three systems, six questions

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | | | |
| C-02 | | | |
| C-03 | | | |
| C-04 | | | |
| C-05 | | | |
| C-06 | | | |
| **agrees with `reference`** | /6 | /6 | /6 |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>

**2. When can BM25 perform better than embeddings?**

>

**3. When can embeddings perform better than BM25?**

>

**4. Why is cosine similarity used with embeddings?**

>

**5. What information is actually sent to an LLM in a RAG system?**

>

**6. Does the LLM search the original documents itself?**

>

**7. What happens if retrieval returns an irrelevant chunk?**

>

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>

**9. Is a vector database required to build a RAG system?**

>

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>

---
# Conclusion

>